In [20]:
import torch
import torch.nn as nn

# Input Image
X = torch.randn(1, 1, 64, 64)

# Dummy Target Mask
target = torch.randint(0, 2, (1, 1, 64, 64)).float()

# Simple U-Net
class SimpleUNet(nn.Module):

    def __init__(self):
        super().__init__()

        # Encoder
        self.enc1 = nn.Sequential(
            nn.Conv2d(1, 8, kernel_size=3, padding=1),
            nn.ReLU()
        )

        self.pool = nn.MaxPool2d(2)

        # Bottleneck
        self.bottleneck = nn.Sequential(
            nn.Conv2d(8, 16, kernel_size=3, padding=1),
            nn.ReLU()
        )

        # Decoder
        self.up = nn.ConvTranspose2d(16, 8, kernel_size=2, stride=2)

        self.dec1 = nn.Sequential(
            nn.Conv2d(16, 8, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(8, 1, kernel_size=1),
            nn.Sigmoid()
        )

    def forward(self, x):

        # Encoder
        e1 = self.enc1(x)

        # Down Sampling
        p = self.pool(e1)

        # Bottleneck
        b = self.bottleneck(p)

        # Up Sampling
        u = self.up(b)

        # Skip Connection
        merge = torch.cat([u, e1], dim=1)

        # Decoder
        out = self.dec1(merge)

        return out

# Dice Loss
def dice_loss(pred, target):

    smooth = 1

    intersection = (pred * target).sum()

    dice = (2 * intersection + smooth) / (pred.sum() + target.sum() + smooth)

    return 1 - dice

# IoU
def iou(pred, target):

    pred = (pred > 0.5).float()

    intersection = (pred * target).sum()

    union = pred.sum() + target.sum() - intersection

    return (intersection + 1e-5) / (union + 1e-5)


# Create Model
model = SimpleUNet()

# Forward Pass
output = model(X)

# Calculate Loss
loss = dice_loss(output, target)

# Calculate IoU
score = iou(output, target)

# Results
print("Input Shape :", X.shape)
print("Output Shape:", output.shape)

print("\nDice Loss :", loss.item())
print("IoU Score :", score.item())

Input Shape : torch.Size([1, 1, 64, 64])
Output Shape: torch.Size([1, 1, 64, 64])

Dice Loss : 0.5009829998016357
IoU Score : 0.3381107449531555
